In [1]:
# Basic Import
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns
# Modelling
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor,AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression, Ridge,Lasso
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import RandomizedSearchCV
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
import warnings


from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier

In [2]:
df = pd.read_csv("../data/raw/diabetes/diabetes_raw.csv")

In [3]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72.0,35,169.5,33.6,0.627,50,1
1,1,85,66.0,29,102.5,26.6,0.351,31,0
2,8,183,64.0,32,169.5,23.3,0.672,32,1
3,1,89,66.0,23,94.0,28.1,0.167,21,0
4,0,137,40.0,35,168.0,43.1,2.288,33,1


In [4]:
X = df.drop("Outcome", axis=1)
y = df["Outcome"]

In [5]:
## Train Test Split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [6]:
X_train

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
60,2,84,70.0,27,102.5,30.1,0.304,21
618,9,112,82.0,24,169.5,28.2,1.282,50
346,1,139,46.0,19,83.0,28.7,0.654,22
294,0,161,50.0,27,102.5,21.9,0.254,65
231,6,134,80.0,37,370.0,46.2,0.238,46
...,...,...,...,...,...,...,...,...
71,5,139,64.0,35,140.0,28.6,0.411,26
106,1,96,122.0,27,102.5,22.4,0.207,27
270,10,101,86.0,37,169.5,45.6,1.136,38
435,0,141,74.5,32,169.5,42.4,0.205,29


In [7]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [9]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
## Now next code 
def evaluate_model(true, predicted):
    accuracy = accuracy_score(true, predicted)
    precision = precision_score(true, predicted, zero_division=0)
    recall = recall_score(true, predicted, zero_division=0)
    f1 = f1_score(true, predicted, zero_division=0)

    return accuracy, precision, recall, f1

In [10]:
models = {
    "Random Forest": RandomForestClassifier(random_state=42),
    "SVM": SVC(),
    "XGBoost": XGBClassifier(random_state=42, eval_metric="logloss"),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "KNN": KNeighborsClassifier(),
    "Naive Bayes": GaussianNB()
}

In [11]:
model_list = []
accuracy_list = []
precision_list = []
recall_list = []
f1_list = []

In [15]:
model_list = []

train_accuracy_list = []
train_precision_list = []
train_recall_list = []
train_f1_list = []

accuracy_list = []
precision_list = []
recall_list = []
f1_list = []

In [16]:
for model_name, model in models.items():

    # Train model
    model.fit(X_train_scaled, y_train)

    # Predictions
    y_train_pred = model.predict(X_train_scaled)
    y_test_pred = model.predict(X_test_scaled)

    # Training metrics
    train_accuracy, train_precision, train_recall, train_f1 = evaluate_model(
        y_train,
        y_train_pred
    )

    # Testing metrics
    test_accuracy, test_precision, test_recall, test_f1 = evaluate_model(
        y_test,
        y_test_pred
    )

    # Store model name
    model_list.append(model_name)

    # Store Training metrics
    train_accuracy_list.append(train_accuracy)
    train_precision_list.append(train_precision)
    train_recall_list.append(train_recall)
    train_f1_list.append(train_f1)

    # Store Testing metrics
    accuracy_list.append(test_accuracy)
    precision_list.append(test_precision)
    recall_list.append(test_recall)
    f1_list.append(test_f1)

In [17]:
results = pd.DataFrame(
    list(zip(
        model_list,

        train_accuracy_list,
        train_precision_list,
        train_recall_list,
        train_f1_list,

        accuracy_list,
        precision_list,
        recall_list,
        f1_list
    )),
    
    columns=[
        "Model Name",

        "Train Accuracy",
        "Train Precision",
        "Train Recall",
        "Train F1",

        "Test Accuracy",
        "Test Precision",
        "Test Recall",
        "Test F1"
    ]
)

results = results.sort_values(
    by="Test Accuracy",
    ascending=False
).reset_index(drop=True)

results

,Model Name,Train Accuracy,Train Precision,Train Recall,Train F1,Test Accuracy,Test Precision,Test Recall,Test F1
0,Random Forest,1.000000,1.000000,1.000000,1.000000,0.883117,0.813559,0.872727,0.842105
1,Gradient Boosting,0.993485,1.000000,0.981221,0.990521,0.870130,0.807018,0.836364,0.821429
2,XGBoost,1.000000,1.000000,1.000000,1.000000,0.863636,0.814815,0.800000,0.807339
3,SVM,0.887622,0.867347,0.798122,0.831296,0.824675,0.759259,0.745455,0.752294
4,KNN,0.864821,0.812500,0.793427,0.802850,0.798701,0.693548,0.781818,0.735043
5,Logistic Regression,0.773616,0.710227,0.586854,0.642674,0.772727,0.692308,0.654545,0.672897
6,Naive Bayes,0.767101,0.676768,0.629108,0.652068,0.766234,0.666667,0.690909,0.678571
